# Vietnamese TTS - Google Colab UI
Chạy các cell bên dưới để khởi động giao diện Gradio cho TTS Engine.

Đảm bảo bạn đã upload hoặc clone source code `tts` và đang đứng tại thư mục gốc của project (có chứa `tts_engine.py`, `app.py`...).

In [ ]:
!pip install gradio piper-tts requests

In [ ]:
import gradio as gr
import json
import os
import zipfile
import time
import datetime
import requests
import subprocess
import urllib.parse

# List local onnx models if they exist
def list_local_voices():
    voices = []
    onnx_dir = "onnx"
    if os.path.exists(onnx_dir):
        for f in os.listdir(onnx_dir):
            if f.endswith(".onnx"):
                name = f.replace(".onnx", "")
                voices.append(name)
    return voices if voices else ["None"]

local_voices = list_local_voices()

def download_model(url):
    try:
        # Check if URL is valid
        parsed = urllib.parse.urlparse(url)
        if not parsed.scheme:
            return None, "URL không hợp lệ."
            
        model_name = os.path.basename(parsed.path)
        if not model_name.endswith('.onnx'):
            return None, "URL model phải trỏ đến file .onnx"
            
        os.makedirs("downloaded_models", exist_ok=True)
        model_path = os.path.join("downloaded_models", model_name)
        config_path = model_path + ".json"
        
        # Download onnx
        if not os.path.exists(model_path):
            print(f"Downloading model from {url}...")
            r = requests.get(url, allow_redirects=True)
            if r.status_code == 200:
                with open(model_path, 'wb') as f:
                    f.write(r.content)
            else:
                return None, f"Lỗi tải model: HTTP {r.status_code}"
                
        # Download config
        config_url = url + ".json"
        if not os.path.exists(config_path):
            print(f"Downloading config from {config_url}...")
            r = requests.get(config_url, allow_redirects=True)
            if r.status_code == 200:
                with open(config_path, 'wb') as f:
                    f.write(r.content)
            else:
                return None, f"Lỗi tải config: HTTP {r.status_code}"
                
        return model_path, None
    except Exception as e:
        return None, f"Lỗi khi tải model: {str(e)}"

def process_tts(input_text, voice_name, model_url, speed, auto_download):
    if not input_text.strip():
        return None, "Vui lòng nhập nội dung văn bản hoặc JSON."

    output_dir = "colab_output"
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Determine model path
    model_path = None
    if model_url and model_url.strip():
        model_path, err = download_model(model_url.strip())
        if err:
            return None, err
    elif voice_name and voice_name != "None":
        model_path = os.path.join("onnx", voice_name + ".onnx")
        if not os.path.exists(model_path):
            return None, f"Không tìm thấy model local: {model_path}"
    else:
        return None, "Vui lòng chọn model local hoặc nhập URL model."

    # Check if input is JSON
    is_batch = False
    items = []
    try:
        data = json.loads(input_text)
        if isinstance(data, list) and len(data) > 0 and "text" in data[0]:
            is_batch = True
            items = data
    except json.JSONDecodeError:
        pass
    
    if not is_batch:
        items = [{"filename": f"output_{timestamp}.wav", "text": input_text.strip()}]
        
    generated_files = []
    
    for item in items:
        text = item.get("text", "").strip()
        filename = item.get("filename", f"output_{time.time()}.wav")
        if not text:
            continue
            
        if not filename.endswith(".wav"):
            filename += ".wav"
            
        filepath = os.path.join(output_dir, filename)
        
        try:
            # piper uses length_scale to control speed. length_scale = 1.0 / speed. 
            # E.g., speed=2.0 -> length_scale=0.5
            length_scale = 1.0 / speed if speed > 0 else 1.0
            cmd = ["piper", "--model", model_path, "--output_file", filepath, "--length_scale", str(length_scale)]
            
            process = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            stdout, stderr = process.communicate(input=text.encode('utf-8'))
            
            if process.returncode == 0 and os.path.exists(filepath):
                generated_files.append(filepath)
            else:
                print(f"Lỗi Piper: {stderr.decode('utf-8', errors='ignore')}")
        except Exception as e:
            print(f"Lỗi khi tạo file {filename}: {e}")
            
    if not generated_files:
        return None, "Không có file nào được tạo ra."
        
    if is_batch or len(generated_files) > 1:
        zip_path = os.path.join(output_dir, f"batch_output_{timestamp}.zip")
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            for f in generated_files:
                zipf.write(f, os.path.basename(f))
        final_output = zip_path
    else:
        final_output = generated_files[0]
        
    if auto_download:
        try:
            from google.colab import files
            files.download(final_output)
        except ImportError:
            print("google.colab not found. Auto-download is only supported in Colab environments.")
            pass
            
    return final_output, f"Hoàn tất. Đã tạo thành công {len(generated_files)} file."

def create_ui():
    with gr.Blocks(title="Vietnamese TTS - Colab UI") as app:
        gr.Markdown("# 🎙️ Piper TTS - Google Colab UI")
        gr.Markdown("Công cụ tổng hợp giọng nói sử dụng **Piper TTS**. Hỗ trợ cả chế độ Văn bản thường và JSON (Batch Mode).")
        
        with gr.Row():
            with gr.Column(scale=2):
                input_text = gr.Textbox(
                    label="Nội dung (Text hoặc JSON)",
                    lines=12,
                    placeholder='''Nhập văn bản bình thường hoặc nhập JSON dạng:\n[\n  {\n    "filename": "audio-scene-1.wav",\n    "text": "script-scene-1_text"\n  }\n]'''
                )
                auto_download = gr.Checkbox(label="Auto Download (Tự động tải xuống khi chạy xong)", value=True)
                
            with gr.Column(scale=1):
                voice_dropdown = gr.Dropdown(
                    choices=local_voices, 
                    value=local_voices[0] if local_voices else None, 
                    label="Giọng đọc (Model Local)"
                )
                model_url = gr.Textbox(
                    label="URL Model (Tùy chọn)",
                    placeholder="https://example.com/model.onnx"
                )
                gr.Markdown("*Ghi chú: Nếu nhập URL Model, hệ thống sẽ tải model từ URL và bỏ qua model local. URL phải trỏ tới file `.onnx` (file `.onnx.json` tương ứng sẽ được tự động tải).*")
                speed = gr.Slider(0.3, 2.0, value=1.0, step=0.1, label="Tốc độ (Speed)")
                
        btn_generate = gr.Button("🚀 Generate TTS", variant="primary")
        
        with gr.Row():
            output_file = gr.File(label="Tệp kết quả (WAV / ZIP)")
            output_msg = gr.Textbox(label="Trạng thái", interactive=False)
            
        btn_generate.click(
            fn=process_tts,
            inputs=[input_text, voice_dropdown, model_url, speed, auto_download],
            outputs=[output_file, output_msg]
        )
        
    return app

app = create_ui()
app.launch(debug=True, inline=True)
